[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# Directories


## What you will be able to do

Find every file matching a pattern anywhere in a folder tree, skip the folders you do not want,
and record what you found before processing any of it.


## The idea

### The problem

Every notebook so far has worked on one file whose name you knew. Real work rarely starts that
way. It starts with a folder someone gave you, containing an unknown number of files, in an
unknown arrangement, some of which are not the ones you want.

Three things go wrong when you process such a folder directly.

**You do not know what is in it.** A loop that reads every `.csv` will happily read a file that
turns out to be a backup, a template, or a copy someone left in a subfolder.

**The order is not what you assume.** The filesystem returns entries in whatever order it likes.
Code that works today because the files happened to come back alphabetically breaks when one is
renamed.

**The folder contains things you did not put there.** `__pycache__`, `.DS_Store`, hidden
directories and editor backups all sit alongside your data and all match `*`.

The fix for all three is the same: look first, record what you found, then process the record.

### The tools

> `Path.iterdir()` lists one folder, one level deep.
>
> `Path.glob(pattern)` lists what matches a pattern, one level deep unless the pattern contains
> `**`.
>
> `Path.rglob(pattern)` searches every level. `rglob("*.csv")` and `glob("**/*.csv")` are the
> same thing.
>
> `os.walk(path)` visits every folder in turn and lets you **prune** branches as you go, which
> the glob functions cannot do.

All four return results in filesystem order, which is not sorted. Wrap anything you rely on in
`sorted()`.

### Building a manifest

The habit worth taking from this notebook is separating discovery from work.

Walk the tree once, collect what you found into a list of records with the path, size and
whatever else you need. Look at that list. Then process it.

The benefit is not tidiness. It is that you can count the files, spot the ones that should not
be there, and report how many of how many you handled, which is the difference between a script
that worked and a script you can defend.

### Where you will meet this

Any task that starts with a folder rather than a file. The **A Small Pipeline** notebook at the
end of this guide does exactly this on a realistic set of files.

### What this notebook covers

- `iterdir`, `glob` and `rglob`, and how far each one looks
- The pattern characters: `*`, `?`, `[]` and `**`
- Why the order is not sorted, and what that breaks
- `os.walk`, and pruning a branch before descending into it
- Reading a file's size without opening it
- Building a manifest, and summarizing it
- Three errors, plus a duplicate that quietly loses half your files

### A first look

Nothing to run yet.

```python
from pathlib import Path

root = Path("project")

for path in sorted(root.rglob("*.csv")):
    print(path, path.stat().st_size)
```

Four lines to find every CSV at any depth. The `sorted` is the part people leave out, and the
part that makes the run repeatable.


## Setup

Five imports and a small folder tree to work on.

- `Path` finds and inspects files, and is most of this notebook
- `os` provides `os.walk`, for the one thing `rglob` cannot do
- `Counter` summarizes the manifest at the end
- `shutil` removes the scratch folder
- `json` writes the manifest out, in the format the **JSON on Disk** notebook covered

**Run this cell before the rest of the notebook.**


In [1]:

from pathlib import Path
from collections import Counter
import os
import json
import shutil

scratch = Path("scratch")
root = scratch / "project"

for folder in ["data/2025", "data/2026", "reports", "data/.hidden", "__pycache__"]:
    (root / folder).mkdir(parents=True, exist_ok=True)

files = [
    "README.md",
    "data/notes.txt",
    "data/2025/jan.csv",
    "data/2025/feb.csv",
    "data/2026/jan.csv",
    "data/.hidden/secret.csv",
    "reports/summary.md",
    "reports/summary.pdf",
    "__pycache__/cache.pyc",
]

for name in files:
    (root / name).write_text("x" * (len(name) * 10))

print(len(files), "files created under", root)


9 files created under scratch/project


## Worked examples

### iterdir: one folder, one level

`iterdir` lists what is directly inside a folder and goes no further.


In [2]:

for path in sorted(root.iterdir()):
    kind = "dir " if path.is_dir() else "file"
    print(f"{kind} {path.relative_to(scratch)}")


file project/README.md
dir  project/__pycache__
dir  project/data
dir  project/reports


Four entries, three of them folders. `iterdir` does not look inside them, which is right when
you want to know what is at this level and wrong when you want the files.

Paths are printed with `relative_to(scratch)` throughout this notebook. The full path would
include wherever this notebook is running, which is different for every reader.

### glob: one level, by pattern


In [3]:

print("markdown at the top:", sorted(str(p.relative_to(root)) for p in root.glob("*.md")))
print("inside data:        ", sorted(str(p.relative_to(root)) for p in root.glob("data/*")))


markdown at the top: ['README.md']
inside data:         ['data/.hidden', 'data/2025', 'data/2026', 'data/notes.txt']


`glob("*.md")` matched only the top level. `README.md` was found and `reports/summary.md` was
not, because the pattern says nothing about subfolders.

### rglob: every level


In [4]:

for path in sorted(root.rglob("*.csv")):
    print(path.relative_to(root))


data/.hidden/secret.csv
data/2025/feb.csv
data/2025/jan.csv
data/2026/jan.csv


Four CSV files at three different depths, found by one call.

Note that `data/.hidden/secret.csv` is in that list. `rglob` descends into hidden folders and
matches hidden files, which surprises people who expect the shell's behavior. If you do not want
them, filter them out yourself.


In [5]:

visible = [p for p in sorted(root.rglob("*.csv"))
           if not any(part.startswith(".") for part in p.parts)]

for path in visible:
    print(path.relative_to(root))


data/2025/feb.csv
data/2025/jan.csv
data/2026/jan.csv


`p.parts` from the **Paths** notebook is the whole path as a tuple, so checking every part
catches a hidden folder anywhere in the chain, not only a hidden filename.

`rglob(pattern)` and `glob("**/" + pattern)` are the same call written two ways.


In [6]:

print(sorted(root.rglob("*.csv")) == sorted(root.glob("**/*.csv")))


True


### The pattern characters

Four of them, and they are not regular expressions.

| Pattern | Matches |
|---|---|
| `*` | any run of characters, including none |
| `?` | exactly one character |
| `[abc]` | one character from the set |
| `**` | this folder and every folder below it |


In [7]:

for pattern in ["*.csv", "?an.csv", "[jf]*.csv", "summary.*"]:
    found = sorted(p.name for p in root.rglob(pattern))
    print(f"{pattern:<12} {found}")


*.csv        ['feb.csv', 'jan.csv', 'jan.csv', 'secret.csv']
?an.csv      ['jan.csv', 'jan.csv']
[jf]*.csv    ['feb.csv', 'jan.csv', 'jan.csv']
summary.*    ['summary.md', 'summary.pdf']


`?an.csv` matched both `jan.csv` files and not `feb.csv`. `[jf]*.csv` matched all three.

These are shell-style patterns, not the regular expressions from the **Regular Expressions**
notebook. `*` here means "anything", where in a regex it means "zero or more of the thing
before it". Mixing them up produces patterns that match nothing and give no clue why.

### Order is not sorted

This is the one that produces a bug you cannot reproduce.


In [8]:

print("as returned:", [p.name for p in root.rglob("*.csv")])
print("sorted:     ", sorted(p.name for p in root.rglob("*.csv")))


as returned: ['secret.csv', 'jan.csv', 'feb.csv', 'jan.csv']
sorted:      ['feb.csv', 'jan.csv', 'jan.csv', 'secret.csv']


The first list is whatever order the filesystem gave. It is stable on one machine and not
guaranteed anywhere, and it changes when files are added, renamed or copied to another disk.

Any code whose result depends on the order, such as taking the first match or writing rows to an
output file, needs `sorted()`. It costs nothing at these sizes and removes a class of bug that
only appears on someone else's machine.


### os.walk, and pruning

`rglob` visits everything and lets you filter afterward. That is fine until the tree contains a
folder you want to avoid entering at all, such as a huge cache directory or a symlinked path.


In [9]:

for folder, subdirs, filenames in os.walk(root):
    print(f"{Path(folder).relative_to(scratch)}  dirs={sorted(subdirs)}  files={sorted(filenames)}")


project  dirs=['__pycache__', 'data', 'reports']  files=['README.md']
project/__pycache__  dirs=[]  files=['cache.pyc']
project/data  dirs=['.hidden', '2025', '2026']  files=['notes.txt']
project/data/.hidden  dirs=[]  files=['secret.csv']
project/data/2025  dirs=[]  files=['feb.csv', 'jan.csv']
project/data/2026  dirs=[]  files=['jan.csv']
project/reports  dirs=[]  files=['summary.md', 'summary.pdf']


`os.walk` yields one tuple per folder: where it is, the folders inside it, and the files inside
it.

The useful part is that `subdirs` is a real list, and **changing it in place changes where the
walk goes next**.


In [10]:

SKIP = {"__pycache__", ".hidden", ".git"}

for folder, subdirs, filenames in os.walk(root):
    subdirs[:] = [d for d in subdirs if d not in SKIP]
    print(f"{Path(folder).relative_to(scratch)}  {sorted(filenames)}")


project  ['README.md']
project/data  ['notes.txt']
project/data/2025  ['feb.csv', 'jan.csv']
project/data/2026  ['jan.csv']
project/reports  ['summary.md', 'summary.pdf']


`__pycache__` and `.hidden` are gone, and the walk never entered them.

`subdirs[:] = [...]` rather than `subdirs = [...]` is the whole trick, and it is the mutation
from the **Lists** notebook doing real work. Assigning a new list would rebind the local name and
leave the walk unchanged; assigning to the slice changes the list `os.walk` is holding.

Use `rglob` for simple searches. Use `os.walk` when you need to not descend somewhere.


### What a file knows about itself

`stat()` asks the filesystem, without opening the file.


In [11]:

target = root / "README.md"
info = target.stat()

print("size:", info.st_size, "bytes")
print("is a file:", target.is_file(), "| is a directory:", target.is_dir())


size: 90 bytes
is a file: True | is a directory: False


`st_mtime` holds the modification time as a number of seconds, and
`datetime.datetime.fromtimestamp` turns it into a date. It is not printed here because the value
would be the moment this notebook last ran, which tells you nothing.

Size is the useful one for a manifest: an empty file, or one far larger than its neighbors, is
usually worth looking at before processing.


### Building a manifest

Discovery first, work second.


In [12]:

manifest = []

for path in sorted(root.rglob("*")):
    if not path.is_file():
        continue
    if any(part.startswith(".") or part == "__pycache__" for part in path.parts):
        continue
    manifest.append({
        "path": str(path.relative_to(root)),
        "suffix": path.suffix,
        "bytes": path.stat().st_size,
    })

for record in manifest:
    print(f"{record['path']:<24} {record['suffix']:<6} {record['bytes']:>5}")


README.md                .md       90
data/2025/feb.csv        .csv     170
data/2025/jan.csv        .csv     170
data/2026/jan.csv        .csv     170
data/notes.txt           .txt     140
reports/summary.md       .md      180
reports/summary.pdf      .pdf     190


Seven files, with the cache and the hidden folder excluded. Nothing has been read yet.

Now the list can be questioned before any work happens.


In [13]:

print("files:      ", len(manifest))
print("by suffix:  ", dict(Counter(r["suffix"] for r in manifest)))
print("total bytes:", sum(r["bytes"] for r in manifest))
print("largest:    ", max(manifest, key=lambda r: r["bytes"])["path"])


files:       7
by suffix:   {'.md': 2, '.csv': 3, '.txt': 1, '.pdf': 1}
total bytes: 1110
largest:     reports/summary.pdf


`Counter` from the **Dictionaries** notebook, `max` with a `key` from the **Functions** notebook.

That summary is worth printing at the start of any script that processes a folder. If it says
three CSV files and you expected twelve, you have found the problem before reading a single
byte.

The manifest is ordinary data, so it can be saved:


In [14]:

record_file = scratch / "manifest.json"
record_file.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(record_file.name, "written,", record_file.stat().st_size, "bytes")
print("first record:", json.loads(record_file.read_text(encoding="utf-8"))[0])


manifest.json written, 558 bytes
first record: {'path': 'README.md', 'suffix': '.md', 'bytes': 90}


Keeping it means a later run can compare against it: which files are new, which changed size,
which have gone. That is most of what a data pipeline does between runs.


### Making folders

`mkdir` on its own creates one folder and fails if the parent is missing or the folder exists.


In [15]:

output = scratch / "output" / "2026" / "march"

output.mkdir(parents=True, exist_ok=True)

print("created:", output.relative_to(scratch), "->", output.is_dir())


created: output/2026/march -> True


`parents=True` makes the whole chain. `exist_ok=True` means running the cell twice is not an
error, which matters in a notebook where cells are re-run constantly.

Without them you get the two errors below.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/07-directories-solutions.ipynb).

**1.** List everything directly inside `root`, sorted, saying whether each is a file or a
folder.


In [16]:
# your code here


**2.** Find every `.md` file at any depth and print its path relative to `root`.


In [17]:
# your code here


**3.** Find every file whose name starts with `s`, at any depth.


In [18]:
# your code here


**4.** Walk the tree with `os.walk`, skipping `__pycache__`, and print each folder with its file
count.


In [19]:
# your code here


**5.** Build a list of records for every `.csv` file, each with its path and size, and print the
total size.


In [20]:
# your code here


**6.** Count how many files share a name with another file somewhere else in the tree.


In [21]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### FileNotFoundError: mkdir will not create the parents


In [22]:

(scratch / "deep" / "nested" / "folder").mkdir()


FileNotFoundError: [Errno 2] No such file or directory: 'scratch/deep/nested/folder'

`No such file or directory`. `mkdir` makes one folder, and `scratch/deep/nested` does not exist
to make it in.

`parents=True` makes the whole chain.


### FileExistsError: it is already there


In [23]:

(scratch / "output").mkdir(parents=True)


FileExistsError: [Errno 17] File exists: 'scratch/output'

`File exists`. Adding `exist_ok=True` makes this a no-op instead, which is what you want almost
every time, and certainly in a notebook.


In [24]:

(scratch / "output").mkdir(parents=True, exist_ok=True)

print("no complaint the second time")


no complaint the second time


### NotADirectoryError: iterdir on a file


In [25]:

list((root / "README.md").iterdir())


NotADirectoryError: [Errno 20] Not a directory: 'scratch/project/README.md'

`Not a directory`. Check with `is_dir()` when the path came from somewhere you do not control,
because a name says nothing about what it is.


### The quiet one: a dictionary keyed by filename


In [26]:

by_name = {}

for path in sorted(root.rglob("*.csv")):
    by_name[path.name] = path.stat().st_size

print("files found by rglob:", len(list(root.rglob("*.csv"))))
print("entries in the dict: ", len(by_name))
print(by_name)


files found by rglob: 4
entries in the dict:  3
{'secret.csv': 230, 'feb.csv': 170, 'jan.csv': 170}


Four files went in and three came out. Nothing raised.

There are two files called `jan.csv`, in `data/2025` and `data/2026`. The second overwrote the
first, because a dictionary key must be unique and the **name** is not.

This is the most common way a folder-processing script loses data, and it produces a result that
looks complete. Key on the full path, which is unique:


In [27]:

by_path = {str(p.relative_to(root)): p.stat().st_size for p in sorted(root.rglob("*.csv"))}

print("entries now:", len(by_path))
for key, size in by_path.items():
    print(f"  {key:<24} {size}")


entries now: 4
  data/.hidden/secret.csv  230
  data/2025/feb.csv        170
  data/2025/jan.csv        170
  data/2026/jan.csv        170


Four entries, and the two `jan.csv` files are distinguishable. Whenever you build a lookup from
files, ask whether the key you chose can collide.


### Cleaning up


In [28]:

shutil.rmtree(scratch)

print("scratch still there:", scratch.exists())


scratch still there: False


## Recap

- `iterdir` lists one level, `glob` matches one level, `rglob` searches every level.
- `rglob(p)` and `glob("**/" + p)` are the same call.
- Patterns are shell-style, not regular expressions: `*`, `?`, `[abc]`, `**`.
- `rglob` descends into hidden folders. Filter them out with a check over `path.parts`.
- Results come back in filesystem order. Wrap anything you depend on in `sorted()`.
- `os.walk` can **prune**: `subdirs[:] = [...]` stops it descending into what you removed.
- `stat().st_size` reads a file's size without opening it.
- Build a manifest before processing, then count and summarize it. A wrong file count is much
  cheaper to find than a wrong result.
- Filenames are not unique across folders. Key on the path.


## What is next

The **Compression and Archives** notebook, which handles the folder that arrived as a single
`.zip`, and reads what is inside it without unpacking it to disk first.


---

&#8592; **Previous:** [Excel Files](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/06-excel-files.ipynb)  &nbsp;·&nbsp;  [Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)  &nbsp;·&nbsp;  **Next:** [Compression and Archives](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/08-compression-and-archives.ipynb) &#8594;
